# Qwen2.5-3B SFT Eval — Disaster Response

Evaluates the **SFT warm-start checkpoint** (`arunms911/disaster-response-sft`) against all 3 difficulties.
Saves scores, per-step logs, and a comparison chart (untuned vs SFT) to `/content/`.

**Prerequisites:**
- Colab secret `HF_TOKEN` set (the SFT repo is private)

Run order: Cell 0 → 1 → 2 → 3 → 4 → 5 → 6

In [ ]:
%%capture
# Clone repo (skips .env and all .gitignore'd files automatically)
!git clone https://github.com/MSArunkutz/disaster-env /content/disaster_env

# Install env dependencies
!pip install -q -r /content/disaster_env/requirements.txt 2>/dev/null || \
    pip install -q fastapi uvicorn pydantic python-dotenv openai websockets httpx

# Install training dependencies
!pip install unsloth transformers accelerate bitsandbytes matplotlib

In [ ]:
import os, sys, json, re
from pathlib import Path

# HF token — needed to pull private SFT repo
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it as a Colab secret (key icon in left sidebar).")

DISASTER_ENV_PATH = Path("/content/disaster_env")
sys.path.insert(0, str(DISASTER_ENV_PATH))

MODEL_NAME   = "arunms911/disaster-response-sft"
MAX_SEQ_LEN  = 2048
LOAD_IN_4BIT = True
MAX_STEPS    = 200

OUTPUT_DIR   = Path("/content")
LOG_FILE     = OUTPUT_DIR / "sft_log.json"
RESULTS_FILE = OUTPUT_DIR / "sft_results.json"
CHART_FILE   = OUTPUT_DIR / "sft_chart.png"

# Baseline numbers from untuned Qwen2.5-3B (for comparison chart)
BASELINE_SCORES = {"easy": 0.0, "medium": 0.0, "hard": 0.216}

print("disaster_env path:", DISASTER_ENV_PATH.resolve())
print("Exists:", DISASTER_ENV_PATH.exists())
print("Model:", MODEL_NAME)
print("Config loaded.")

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = LOAD_IN_4BIT,
    token          = HF_TOKEN,
)
FastLanguageModel.for_inference(model)
print("Loaded (SFT checkpoint):", MODEL_NAME)

In [ ]:
def extract_action_json(text: str):
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    m = re.search(r"(\{.*\})", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    return None

In [ ]:
from server.disaster_env_environment import DisasterEnvEnvironment
from models import DisasterAction
from graders import compute_score


def run_sft_episode(difficulty: str) -> tuple:
    """Run one episode, return (score, step_logs)."""
    env = DisasterEnvEnvironment(difficulty=difficulty)
    obs = env.reset()
    step_logs = []

    for _ in range(MAX_STEPS):
        if obs.done:
            break

        prompt_text = (
            f"Step {obs.current_step}/{obs.max_steps}\n"
            f"{obs.zones_summary}\n{obs.resources_summary}\n{obs.available_actions}\n"
            "Respond with a JSON deployments object."
        )
        inputs  = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
        text    = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        action_data = extract_action_json(text)
        entry = {
            "step":          obs.current_step,
            "raw_output":    text[:400],
            "parse_success": action_data is not None,
        }

        if action_data and "deployments" in action_data:
            try:
                obs = env.step(DisasterAction(**action_data))
            except Exception as e:
                entry["error"] = str(e)
                step_logs.append(entry)
                break
        else:
            entry["parse_failed"] = True
            step_logs.append(entry)
            break

        step_logs.append(entry)

    score = compute_score(env)
    return score, step_logs


all_logs = {}
results  = {}

for diff in ["easy", "medium", "hard"]:
    print(f"Running {diff}...")
    score, logs = run_sft_episode(diff)
    results[diff]  = score
    all_logs[diff] = logs
    parse_ok = sum(1 for s in logs if s.get("parse_success"))
    print(f"  {diff:8s}  score={score:.4f}  steps={len(logs)}  parsed={parse_ok}/{len(logs)}")

avg = sum(results.values()) / 3
results["average"] = round(avg, 4)
print(f"\nAverage: {avg:.4f}")

# Delta vs untuned baseline
print("\n--- Delta vs untuned baseline ---")
for d in ["easy", "medium", "hard"]:
    delta = results[d] - BASELINE_SCORES[d]
    print(f"  {d:8s}: {BASELINE_SCORES[d]:.4f} → {results[d]:.4f}  ({'+' if delta >= 0 else ''}{delta:.4f})")

In [ ]:
with open(RESULTS_FILE, "w") as f:
    json.dump({"model": MODEL_NAME, "results": results}, f, indent=2)
print("Saved:", RESULTS_FILE)

with open(LOG_FILE, "w") as f:
    json.dump(all_logs, f, indent=2)
print("Saved:", LOG_FILE)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

difficulties = ["easy", "medium", "hard"]
sft_scores      = [results[d] for d in difficulties]
baseline_scores = [BASELINE_SCORES[d] for d in difficulties]

x     = np.arange(len(difficulties))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
b1 = ax.bar(x - width / 2, baseline_scores, width, label="Untuned (baseline)", color="#64748b")
b2 = ax.bar(x + width / 2, sft_scores,      width, label="SFT warm-start",     color="#3b82f6")

ax.set_xticks(x)
ax.set_xticklabels(difficulties)
ax.set_ylim(0, 1.1)
ax.set_ylabel("compute_score")
ax.set_title("Qwen2.5-3B: Untuned vs SFT warm-start")
ax.legend()

for bar in [*b1, *b2]:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.4f}",
            ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(CHART_FILE, dpi=150)
plt.show()
print("Saved:", CHART_FILE)
print("\n--- Final SFT Scores ---")
for d in difficulties:
    delta = results[d] - BASELINE_SCORES[d]
    print(f"  {d:8s}: {results[d]:.4f}  ({'+' if delta >= 0 else ''}{delta:.4f} vs untuned)")
print(f"  {'average':8s}: {results['average']:.4f}")